# Ground and RSO Imaging on One Satellite

This example configures **one imager** with ground and space-to-space imaging actions and observations. Two passive spacecraft serve as RSO targets. We task one ground target and both RSOs by ID, then downlink through their shared storage unit and transmitter.

The nearby targets, permissive pointing tolerance, disabled RSO illumination gating, and ground-station elevation limit of -90 degrees keep this configuration demonstration short. They are demonstration settings rather than mission constraints. The ground scene retains spare candidates after acquisition so its observation continues to have upcoming opportunities.

In [1]:
from pprint import pprint
from typing import ClassVar

import numpy as np
from Basilisk.utilities import orbitalMotion

from bsk_rl import ConstellationTasking, act, data, obs, sats, scene
from bsk_rl.scene.targets import Target
from bsk_rl.sim import dyn, fsw

EPOCH = "2026 OCT 01 00:00:00.000 (UTC)"

/Users/lorenzzo/Local/repo_updates/issue_366/.venv_366/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Define the mixed imager

`ImagingSatellite` supplies ground opportunity selection and acquisition tracking. `SpaceToSpaceImagingDynModel` and `SpaceToSpaceImagingFSWModel` retain ground imaging and add RSO access and a separate pointing task. Composing `GroundStationDynModel` also enables downlink.

The observation combines resource state, two ground-opportunity rows, and two RSO-candidate rows. RSO rows include a `valid` flag for empty candidate slots. The dictionary format makes these groups easy to inspect; use `obs_type=np.ndarray` for a flattened policy input.

| Integer action | Meaning |
| --- | --- |
| 0 | Charge |
| 1 | Downlink |
| 2–3 | Image a ground candidate |
| 4–5 | Image an RSO candidate |

IDs and target objects can also select either imaging mode. ID dispatch works with either ordering of `Image` and `ImageRSO`; if both catalogues contain the same ID, pass a target object to disambiguate it.

In [2]:
class MixedImager(sats.ImagingSatellite):
    """Share the instrument, attitude controller, storage, and transmitter."""

    dyn_type = (dyn.SpaceToSpaceImagingDynModel, dyn.GroundStationDynModel)
    fsw_type = fsw.SpaceToSpaceImagingFSWModel
    observation_spec: ClassVar[list[obs.Observation]] = [
        obs.SatProperties(
            dict(prop="storage_level_fraction"),
            dict(prop="battery_charge_fraction"),
        ),
        obs.OpportunityProperties(
            dict(prop="priority"),
            dict(prop="opportunity_open", norm=60),
            dict(prop="opportunity_close", norm=60),
            n_ahead_observe=2,
            name="ground_targets",
        ),
        obs.RSOTargetProperties(
            dict(prop="valid"),
            dict(prop="priority"),
            dict(prop="target_distance", norm=1e3),
            dict(prop="target_illumination_factor"),
            n_ahead_observe=2,
        ),
    ]
    # Integer indices: charge=0, downlink=1, ground candidates=2..3,
    # RSO candidates=4..5.
    action_spec: ClassVar[list[act.Action]] = [
        act.Charge(duration=10),
        act.Downlink(duration=60),
        act.Image(n_ahead_image=2, max_duration=30),
        act.ImageRSO(
            n_ahead_image=2,
            max_duration=30,
            min_pointing_hold_s=2,
            require_illumination_during_hold=False,
        ),
    ]

## Define the target models

The passive RSO spacecraft propagate normally and expose only a drift action. They participate in the RSO scene without producing imaging rewards.

For this short example, three ground candidates are placed below the imager at the initial epoch. The scene updates their planet-fixed positions after the simulator knows the Earth's orientation and before ground access windows are calculated.

In [3]:
class PassiveTarget(sats.Satellite):
    """Propagate a catalog spacecraft without giving it imaging actions."""

    dyn_type = dyn.RSOTargetDynModel
    fsw_type = fsw.FSWModel
    observation_spec: ClassVar[list[obs.Observation]] = [obs.Time()]
    action_spec: ClassVar[list[act.Action]] = [act.Drift(duration=1e9)]


class DemoGroundTargets(scene.UniformTargets):
    """Place three ground candidates below the imager at the initial epoch."""

    def regenerate_targets(self):
        self.targets = [
            Target(
                f"ground_{index}",
                [orbitalMotion.REQ_EARTH * 1e3, 0, 0],
                priority=3,
            )
            for index in range(3)
        ]

    def reset_post_sim_init(self):
        # Account for the realized Earth orientation. Access registration retains
        # each position array, so update it before satellite windows are generated.
        imager = self.satellites[0]
        position_P = imager.dynamics.world.PN @ imager.dynamics.r_BN_N
        for target in self.targets:
            target.r_LP_P[:] = self.radius * position_P / np.linalg.norm(position_P)

## Configure the mission

The imager has a **total capacity of 5,000 bits**. Each image contains 1,000 bits, and the shared transmitter drains data at 100 bits/s. The RSO catalogue explicitly binds each ID to a passive target spacecraft.

In [4]:
imager = MixedImager(
    "imager",
    # A dictionary keeps the two target observation groups easy to inspect.
    # Use obs_type=np.ndarray to flatten them into a policy input vector.
    obs_type=dict,
    sat_args=dict(
        rN=[7e6, 0, 0],
        vN=[0, 7500, 0],
        oe=None,
        sigma_init=[0, 0, 0],
        omega_init=[0, 0, 0],
        wheelSpeeds=[0.0, 0.0, 0.0],
        imageAttErrorRequirement=2.0,
        instrumentBaudRate=1000,  # Image size [bits], for either imaging mode.
        transmitterBaudRate=-100,  # Downlink [bits/s], shared across partitions.
        dataStorageCapacity=5000,  # Total capacity, not capacity per partition.
        storageInit=0,
        batteryStorageCapacity=1e9,
        storedCharge_Init=1e9,
    ),
)

catalog = scene.RSOTargetCatalog(
    (
        scene.RSOTarget(
            "rso_a", "debris_a", (7e6 + 1000, 0, 0), (0, 7500, 0), priority=2
        ),
        scene.RSOTarget(
            "rso_b", "debris_b", (7e6 + 2000, 0, 0), (0, 7500, 0), priority=1
        ),
    ),
    EPOCH,
)

## Bind scenarios and reward channels

The imager belongs to both scenes and both reward channels. Scenario participants also include the physical RSO target spacecraft; reward-channel assignments contain only the imager. `scenario_mapping` links each reward channel to its corresponding scene.

Ground images earn their priority once on acquisition. For this example, RSO callbacks award **90% of priority on acquisition and 10% on complete delivery**. The generic RSO rewarder defaults to delivery-only credit; the 90/10 split is an explicit choice here.

No custom reset or manual buffer allocation is needed. The library keeps one ground partition at index zero, followed by one named partition per RSO, within the same storage unit. All partitions share its total capacity.

In [5]:
mixed_scene = scene.MixedScenario(
    scenarios={
        "ground_scene": DemoGroundTargets(3),
        "rso_scene": scene.RSOTargets(catalog, [imager.name]),
    },
    satellite_mapping={
        "ground_scene": [imager.name],
        "rso_scene": [
            imager.name,
            *(target.satellite_name for target in catalog.targets),
        ],
    },
)

mixed_reward = data.MixedReward(
    rewarders={
        "ground_images": data.UniqueImageReward(),
        "rso_images": data.RSOImageReward(
            acquisition_reward_fn=lambda record, target: 0.9 * target.priority,
            delivery_reward_fn=lambda record, target: 0.1 * target.priority,
            quality_threshold=0,
        ),
    },
    satellite_mapping={"ground_images": [imager.name], "rso_images": [imager.name]},
    scenario_mapping={"ground_images": "ground_scene", "rso_images": "rso_scene"},
)

## Construct the environment

Only the imager has mixed actions. The two additional satellites are its passive RSO targets. The environment advances at most one second per step, making the capture and delivery boundaries easy to inspect.

In [6]:
env = ConstellationTasking(
    satellites=[
        imager,
        *(PassiveTarget(target.satellite_name) for target in catalog.targets),
    ],
    scenario=mixed_scene,
    rewarder=mixed_reward,
    world_args=dict(utc_init=EPOCH, gsMinimumElevation=np.radians(-90)),
    time_limit=180,
    max_step_duration=1,
    sim_rate=1,
    log_level="ERROR",
)

## Reset and inspect the observation and action spaces

The environment copies its configured participants and scenes. After reset, use the environment's satellite and target objects. The printed partition names show the ground buffer and the two RSO buffers sharing one storage unit.

In [7]:
seed = 0
observations, _ = env.reset(seed=seed)
imager = next(satellite for satellite in env.satellites if satellite.name == "imager")
stores = imager.data_store.data_stores
rso_scene = env.scenario.scenarios["rso_scene"]
ground = env.scenario.scenarios["ground_scene"].targets[0]
partitions = list(
    imager.dynamics.storageUnit.storageUnitDataOutMsg.read().storedDataName
)

print("Observation groups:", list(observations[imager.name]))
print("Ground candidate rows:")
pprint(observations[imager.name]["ground_targets"])
print("RSO candidate rows:")
pprint(observations[imager.name]["rso_targets"])
print("Integer actions:")
pprint(dict(enumerate(imager.action_description)))
print("Shared storage partitions:", partitions)

Observation groups: ['sat_props', 'ground_targets', 'rso_targets']
Ground candidate rows:
{'ground_targets_0': {'opportunity_close_normd': 1.4230079841995078,
                      'opportunity_open_normd': np.float64(0.0),
                      'priority': 3.0},
 'ground_targets_1': {'opportunity_close_normd': 1.4230079841995078,
                      'opportunity_open_normd': np.float64(0.0),
                      'priority': 3.0}}
RSO candidate rows:
{'rso_targets_0': {'priority': 2.0,
                   'target_distance': 1.0,
                   'target_illumination_factor': 0.0,
                   'valid': 1.0},
 'rso_targets_1': {'priority': 1.0,
                   'target_distance': 2.0,
                   'target_illumination_factor': 0.0,
                   'valid': 1.0}}
Integer actions:
{0: 'action_charge',
 1: 'action_downlink',
 2: 'action_image_0',
 3: 'action_image_1',
 4: 'image_rso_0',
 5: 'image_rso_1'}
Shared storage partitions: ['image_buffer', 'rso_f9ec1e33845a6f42

## Step until a scripted task completes

`None` continues the current action. The helper below starts each requested task once, keeps the passive spacecraft drifting, and records total storage usage while the simulation advances. Its step limit reports a failure if a demonstration task cannot finish.

In [8]:
history = []


def advance_until(action, finished, max_steps=60):
    for index in range(max_steps):
        _, _, terminated, truncated, _ = env.step(
            {
                satellite.name: (action if index == 0 else None)
                if satellite is imager
                else (0 if env.simulator.sim_time == 0 else None)
                for satellite in env.satellites
            }
        )
        history.append(
            dict(
                time_s=float(env.simulator.sim_time),
                storage_bits=float(imager.dynamics.storage_level),
            )
        )
        if finished():
            return
        if terminated[imager.name] or truncated[imager.name]:
            raise RuntimeError("The example ended before completing its scripted task.")
    raise RuntimeError(
        f"The scripted task {action!r} did not finish in {max_steps} steps."
    )

## Acquire a ground target by ID

This passes the ground target's ID through the ordinary action builder. The acquisition is detected from an increase in the ground buffer, with no search through the RSO partitions.

In [9]:
advance_until(ground.id, lambda: ground in stores["ground_images"].data.imaged)
print("Acquired ground target by ID:", ground.id)
print(
    "Ground acquisition reward:",
    env.rewarder.rewarders["ground_images"].cum_reward[imager.name],
)

Acquired ground target by ID: ground_0_4976925184
Ground acquisition reward: 3.0


## Acquire the RSO targets by ID

The same action interface routes each RSO ID to `ImageRSO`. Switching modes cancels any previous acquisition hold and selects the RSO instrument command source. Each image is recorded only after a complete product has reached its target's storage partition.

In [10]:
for target_id in rso_scene.targets_by_id:
    advance_until(
        target_id,
        lambda target_id=target_id: any(
            record.target_id == target_id
            for record in stores["rso_images"].data.captures.values()
        ),
    )
    print("Acquired RSO target by ID:", target_id)
print(
    "RSO acquisition reward:",
    env.rewarder.rewarders["rso_images"].cum_reward[imager.name],
)

Acquired RSO target by ID: rso_a
Acquired RSO target by ID: rso_b
RSO acquisition reward: 2.7


## Inspect shared storage before downlink

There should now be three 1,000-bit images: one in the ground buffer and one in each RSO partition. Their combined usage is 3,000 bits of the single 5,000-bit capacity.

In [11]:
message = imager.dynamics.storageUnit.storageUnitDataOutMsg.read()
pprint(dict(zip(message.storedDataName, message.storedData)))
print("Total storage [bits]:", imager.dynamics.storage_level)
print("Storage fraction:", imager.dynamics.storage_level_fraction)

{'image_buffer': 1000.0,
 'rso_3b64ca2a775eaf93edec3dce613b3749': 1000.0,
 'rso_f9ec1e33845a6f42682e0ef304bc4b6e': 1000.0}
Total storage [bits]: 3000.0
Storage fraction: 0.6


## Downlink and inspect the rewards

Integer action `1` selects downlink. It drains the partitions through the shared transmitter. Partial RSO transmissions remain in the product ledger; delivery credit is awarded only when all bits of an image have left storage.

Expected results are one ground image, two acquired and delivered RSO images, zero final storage, and cumulative rewards of **3 for ground imaging and 3 for RSO imaging**. The full sequence takes 40 simulated seconds in this configuration.

In [12]:
advance_until(1, lambda: imager.dynamics.storage_level == 0)
summary = dict(
    ground_images=len(stores["ground_images"].data.imaged),
    rso_captures=len(stores["rso_images"].data.captures),
    rso_deliveries=len(stores["rso_images"].data.deliveries),
    storage_partitions=partitions,
    peak_storage_bits=max(row["storage_bits"] for row in history),
    final_storage_bits=float(imager.dynamics.storage_level),
    reward_by_channel={
        channel: float(rewarder.cum_reward[imager.name])
        for channel, rewarder in env.rewarder.rewarders.items()
    },
    elapsed_s=float(env.simulator.sim_time),
)
pprint(summary)

{'elapsed_s': 40.0,
 'final_storage_bits': 0.0,
 'ground_images': 1,
 'peak_storage_bits': 3000.0,
 'reward_by_channel': {'ground_images': 3.0, 'rso_images': 3.0000000000000004},
 'rso_captures': 2,
 'rso_deliveries': 2,
 'storage_partitions': ['image_buffer',
                        'rso_f9ec1e33845a6f42682e0ef304bc4b6e',
                        'rso_3b64ca2a775eaf93edec3dce613b3749']}
